# Searching & Planning Demo (Jupyter)



In [ ]:
import numpy as np
import heapq
from collections import deque
import time

import matplotlib.pyplot as plt

# Widgets are required for Play/Slider UI
try:
    import ipywidgets as W
except Exception as e:
    W = None
    print("ipywidgets is required for interactive playback.")
    print("Install it with: pip install -q ipywidgets")
    print("Error was:", repr(e))

try:
    from IPython.display import display
except Exception:
    display = print


## Animation notes (avoid flashing + “no animation” issues)

This notebook uses **ipywidgets** for the Play/Slider UI and renders frames into a **widgets.Image**.
That means it works even if your Matplotlib backend is the default inline backend.

If you **see only a static image**:
- First check that the **Play + frame slider** widgets appear.
- If widgets do not appear, install/enable them:
  - `pip install -q ipywidgets`
  - JupyterLab sometimes also needs: `pip install -q jupyterlab_widgets`

Optional (faster but not required): if `ipympl` is available, you can enable `%matplotlib widget` for true canvas updates.


In [ ]:
# Try to enable smooth, non-flickery plotting in Jupyter
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
    print("Matplotlib backend: widget (smooth updates enabled)")
except Exception as e:
    print("Could not enable '%matplotlib widget'.")
    print("If you want smooth playback, install ipympl then rerun this cell:")
    print("  pip install -q ipympl")
    print("Error was:", repr(e))

## Part A — Grid environments (maze + Manhattan city)

Conventions:
- `grid[y, x] = 1` is **wall/building**, `0` is **free**.
- 4-neighbor moves (up/down/left/right).
- **Manhattan heuristic** is admissible for unit-cost 4-neighbor moves.

### Why BFS and Dijkstra can look identical
If every move cost is `1`, then Dijkstra’s priority (`g`) equals BFS depth. They expand nodes in the same layers.

To make them differ, add **non-uniform costs** (traffic/terrain/turn penalties). Then:
- BFS minimizes **steps**.
- Dijkstra minimizes **total cost**.


In [ ]:
def gen_maze(h, w, seed=0):
    """Maze via recursive backtracker carving. Output: 0=free, 1=wall."""
    rng = np.random.default_rng(seed)
    h = h if h % 2 == 1 else h - 1
    w = w if w % 2 == 1 else w - 1
    grid = np.ones((h, w), dtype=np.uint8)

    def neighbors2(y, x):
        for dy, dx in [(-2,0),(2,0),(0,-2),(0,2)]:
            ny, nx = y + dy, x + dx
            if 1 <= ny < h-1 and 1 <= nx < w-1:
                yield ny, nx, dy//2, dx//2

    sy, sx = 1, 1
    grid[sy, sx] = 0
    stack = [(sy, sx)]
    visited = {(sy, sx)}

    while stack:
        y, x = stack[-1]
        cand = [(ny, nx, wy, wx) for ny, nx, wy, wx in neighbors2(y, x) if (ny, nx) not in visited]
        if not cand:
            stack.pop()
            continue
        ny, nx, wy, wx = cand[rng.integers(len(cand))]
        grid[y + wy, x + wx] = 0
        grid[ny, nx] = 0
        visited.add((ny, nx))
        stack.append((ny, nx))
    return grid


def gen_manhattan_city(h, w, block=7, road=2, alley_prob=0.07, seed=0):
    """Manhattan-like city blocks separated by orthogonal roads. 0=road/free, 1=building."""
    rng = np.random.default_rng(seed)
    grid = np.ones((h, w), dtype=np.uint8)
    step = block + road

    # Horizontal roads
    for y in range(0, h, step):
        grid[y:y+road, :] = 0

    # Vertical roads
    for x in range(0, w, step):
        grid[:, x:x+road] = 0

    # Occasional alleys inside blocks -> shortcuts / edge cases
    for y0 in range(0, h, step):
        for x0 in range(0, w, step):
            by0, by1 = y0 + road, min(y0 + step, h)
            bx0, bx1 = x0 + road, min(x0 + step, w)
            if by1 - by0 <= 2 or bx1 - bx0 <= 2:
                continue
            if rng.random() < alley_prob:
                ax = rng.integers(bx0+1, bx1-1)
                grid[by0:by1, ax:ax+1] = 0
            if rng.random() < alley_prob:
                ay = rng.integers(by0+1, by1-1)
                grid[ay:ay+1, bx0:bx1] = 0
    return grid


def sample_start_goal(grid, seed=0):
    rng = np.random.default_rng(seed)
    free = np.argwhere(grid == 0)
    if len(free) < 2:
        raise ValueError("Not enough free cells for start/goal.")
    s_idx, g_idx = rng.choice(len(free), size=2, replace=False)
    s = tuple(map(int, free[s_idx]))
    g = tuple(map(int, free[g_idx]))
    return s, g


def make_costs(grid, seed=0, p=0.15, lo=2, hi=8):
    """Cost-to-enter map: default 1, but p-fraction of free cells get higher costs."""
    rng = np.random.default_rng(seed)
    cost = np.ones_like(grid, dtype=np.float32)
    free = np.argwhere(grid == 0)
    m = int(len(free) * p)
    if m <= 0:
        return cost
    idx = rng.choice(len(free), size=m, replace=False)
    for y, x in free[idx]:
        cost[y, x] = float(rng.integers(lo, hi+1))
    return cost

### Search algorithms (BFS / DFS / Dijkstra / A*) on the grid

- `weighted=False`: unit-cost edges (BFS ~ Dijkstra).
- `weighted=True`: Dijkstra/A* use `cell_cost[y,x]` as cost-to-enter. BFS/DFS still use steps only.

To keep playback smooth and memory-light, the search records:
- `explored_order`: list of expanded nodes in time order
- `steps`: periodic snapshots containing `current` node and a **sample** of the frontier


In [ ]:
def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def neighbors4(grid, node):
    y, x = node
    for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]:
        ny, nx = y+dy, x+dx
        if 0 <= ny < grid.shape[0] and 0 <= nx < grid.shape[1] and grid[ny, nx] == 0:
            yield (ny, nx)

def reconstruct_path(came_from, start, goal):
    if goal == start:
        return [start]
    if goal not in came_from:
        return None
    cur = goal
    path = [cur]
    while cur != start:
        cur = came_from[cur]
        path.append(cur)
    path.reverse()
    return path

def path_cost(path, cell_cost=None):
    if not path:
        return None
    if cell_cost is None:
        return float(len(path) - 1)
    c = 0.0
    for (y, x) in path[1:]:  # cost-to-enter (exclude start)
        c += float(cell_cost[y, x])
    return c

def search_grid_with_steps(grid, start, goal, algo="astar",
                           cell_cost=None, record_every=20,
                           frontier_sample_n=1200, max_expand=2_000_000):
    """Return (steps, explored_order, came_from, found, stats).

    steps: list of dicts: {k, cur, frontier}
      - k: number of expanded nodes so far (len(explored_order))
      - frontier: sampled nodes in open/queue/stack (for visualization)

    For BFS/DFS we treat the frontier as the queue/stack content.
    For Dijkstra/A* we treat the frontier as the open set keys.
    """
    t0 = time.perf_counter()
    came_from = {}
    explored_set = set()
    explored_order = []
    steps = []
    expanded = 0

    weighted = cell_cost is not None

    # ---------- BFS / DFS ----------
    if algo in ("bfs", "dfs"):
        if algo == "bfs":
            frontier_ds = deque([start])
            pop_fn = frontier_ds.popleft
            push_fn = frontier_ds.append
        else:
            frontier_ds = [start]
            pop_fn = frontier_ds.pop
            push_fn = frontier_ds.append

        frontier_set = {start}
        it = 0

        while frontier_ds and expanded < max_expand:
            cur = pop_fn()
            frontier_set.discard(cur)

            if cur in explored_set:
                continue

            explored_set.add(cur)
            explored_order.append(cur)
            expanded += 1

            if expanded % record_every == 0:
                # snapshot frontier content (limited)
                fs = list(frontier_set)
                if len(fs) > frontier_sample_n:
                    fs = fs[:frontier_sample_n]
                steps.append({"k": len(explored_order), "cur": cur, "frontier": fs})

            if cur == goal:
                dt = time.perf_counter() - t0
                steps.append({"k": len(explored_order), "cur": cur, "frontier": []})
                return steps, explored_order, came_from, True, {"expanded": expanded, "time_s": dt}

            for nb in neighbors4(grid, cur):
                if nb in explored_set or nb in frontier_set:
                    continue
                came_from[nb] = cur
                frontier_set.add(nb)
                push_fn(nb)
            it += 1

        dt = time.perf_counter() - t0
        return steps, explored_order, came_from, False, {"expanded": expanded, "time_s": dt}

    # ---------- Dijkstra / A* ----------
    if algo not in ("dijkstra", "astar"):
        raise ValueError("algo must be one of: bfs, dfs, dijkstra, astar")

    # admissible scaling if weighted (min step cost)
    if cell_cost is None:
        min_step = 1.0
    else:
        # free cells have >=1, walls set to 0
        positive = cell_cost[cell_cost > 0]
        min_step = float(np.min(positive)) if positive.size else 1.0

    def h(node):
        if algo == "dijkstra":
            return 0.0
        return manhattan(node, goal) * min_step

    def f(node, g_cost):
        if algo == "dijkstra":
            return g_cost
        return g_cost + h(node)  # astar

    g = {start: 0.0}
    open_best = {start: f(start, 0.0)}  # membership + best f known
    heap = []
    tie = 0
    heapq.heappush(heap, (open_best[start], tie, start))

    while heap and expanded < max_expand:
        _, _, cur = heapq.heappop(heap)

        if cur in explored_set:
            continue
        if cur not in open_best:
            continue

        del open_best[cur]
        explored_set.add(cur)
        explored_order.append(cur)
        expanded += 1

        if expanded % record_every == 0:
            fs = list(open_best.keys())
            if len(fs) > frontier_sample_n:
                fs = fs[:frontier_sample_n]
            steps.append({"k": len(explored_order), "cur": cur, "frontier": fs})

        if cur == goal:
            dt = time.perf_counter() - t0
            steps.append({"k": len(explored_order), "cur": cur, "frontier": list(open_best.keys())[:min(len(open_best), frontier_sample_n)]})
            return steps, explored_order, came_from, True, {"expanded": expanded, "time_s": dt}

        for nb in neighbors4(grid, cur):
            if nb in explored_set:
                continue
            step_cost = 1.0 if cell_cost is None else float(cell_cost[nb[0], nb[1]])
            new_g = g[cur] + step_cost

            if nb not in g or new_g < g[nb]:
                g[nb] = new_g
                came_from[nb] = cur
                tie += 1
                fs = f(nb, new_g)
                heapq.heappush(heap, (fs, tie, nb))
                open_best[nb] = fs

    dt = time.perf_counter() - t0
    steps.append({"k": len(explored_order), "cur": explored_order[-1] if explored_order else start,
                  "frontier": list(open_best.keys())[:min(len(open_best), frontier_sample_n)]})
    return steps, explored_order, came_from, False, {"expanded": expanded, "time_s": dt}


### Smooth interactive playback (no `clear_output`)

This viewer:
- draws the grid once
- updates scatter points in-place
- supports Play + Slider

Notes:
- If you stay on `%matplotlib inline`, you may still see mild redraw artifacts.
- `%matplotlib widget` (ipympl) is the intended mode.


In [ ]:
def interactive_grid_player(
    grid, start, goal, steps, explored_order, path=None,
    title="",
    interval_ms=180,
    explored_tail=7000,
    dpi=90, figsize=(5.5, 5.5), img_px=540,
    explored_alpha=0.55, frontier_alpha=0.85,
    explored_color="tab:blue", frontier_color="gold",
    info=None,
    cell_cost=None,
    show_traffic=None, traffic_alpha=0.55, traffic_cmap="YlOrRd",
    show=False,
):
    if W is None:
        raise RuntimeError("ipywidgets is not installed. Install with: pip install -q ipywidgets")

    import io
    import numpy as np
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

    # auto: traffic overlay ON if a cost map is provided
    if show_traffic is None:
        show_traffic = (cell_cost is not None)

    fig = Figure(figsize=figsize, dpi=dpi)
    FigureCanvas(fig)
    ax = fig.add_subplot(1, 1, 1)

    ax.imshow(grid, interpolation="nearest", cmap="gray_r")
    ax.set_xticks([]); ax.set_yticks([])

    # --- Traffic overlay (static) ---
    if show_traffic and (cell_cost is not None):
        traffic = cell_cost.astype(float).copy()
        traffic[grid == 1] = np.nan  # hide buildings
        vmin = np.nanmin(traffic)
        vmax = np.nanmax(traffic)
        ax.imshow(
            traffic,
            interpolation="nearest",
            cmap=traffic_cmap,
            alpha=traffic_alpha,
            vmin=vmin,
            vmax=vmax
        )

    explored_sc = ax.scatter([], [], marker="s", s=14, alpha=explored_alpha,
                             linewidths=0, c=explored_color)
    frontier_sc = ax.scatter([], [], marker="s", s=14, alpha=frontier_alpha,
                             linewidths=0, c=frontier_color)
    cur_sc      = ax.scatter([], [], marker="s", s=75, alpha=1.0,
                             edgecolors="k", linewidths=0.7)

    ax.scatter([start[1]], [start[0]], marker="s", s=95, alpha=1.0, edgecolors="k", linewidths=0.8)
    ax.scatter([goal[1]],  [goal[0]],  marker="s", s=95, alpha=1.0, edgecolors="k", linewidths=0.8)

    final_line, = ax.plot([], [], linestyle="--", linewidth=3, color="red", zorder=25)
    final_line.set_visible(False)

    def set_offsets(sc, pts):
        if not pts:
            sc.set_offsets(np.empty((0, 2)))
            return
        xs = [p[1] for p in pts]
        ys = [p[0] for p in pts]
        sc.set_offsets(np.c_[xs, ys])

    def fig_png_bytes():
        buf = io.BytesIO()
        fig.canvas.draw()
        fig.canvas.print_png(buf)
        return buf.getvalue()

    slider = W.IntSlider(value=0, min=0, max=max(0, len(steps)-1), step=1, description="frame")
    play = W.Play(value=0, min=0, max=slider.max, step=1, interval=interval_ms)
    W.jslink((play, "value"), (slider, "value"))

    img = W.Image(value=b"", format="png")
    img.layout = W.Layout(width=f"{img_px}px", height=f"{img_px}px")
    status = W.HTML()

    def format_info_block(i, k):
        lines = []
        if title:
            lines.append(title)
        if isinstance(info, dict):
            for kk, vv in info.items():
                if vv is not None:
                    lines.append(f"{kk}={vv}")
        elif isinstance(info, str) and info.strip():
            lines.extend(info.strip().splitlines())
        lines.append(f"frame={i}/{len(steps)-1}")
        lines.append(f"expanded~{k}")
        return "<pre style='margin:0; font-family:monospace; font-size:12px; line-height:1.25;'>" + \
               "\n".join(lines) + "</pre>"

    def render(i):
        # Frame 0: only start/goal + current=start
        if i == 0:
            set_offsets(explored_sc, [])
            set_offsets(frontier_sc, [])
            set_offsets(cur_sc, [start])
            final_line.set_visible(False)
            img.value = fig_png_bytes()
            status.value = format_info_block(i, 0)
            return

        s = steps[i]
        k = s["k"]
        cur = s["cur"]
        frontier = s["frontier"]

        explored = explored_order[:k]
        if explored_tail and len(explored) > explored_tail:
            explored = explored[-explored_tail:]

        set_offsets(explored_sc, explored)
        set_offsets(frontier_sc, frontier)
        set_offsets(cur_sc, [cur])

        # Final path only at the end
        is_done = (i == slider.max)
        if path is not None and is_done:
            xs = [p[1] for p in path]
            ys = [p[0] for p in path]
            final_line.set_data(xs, ys)
            final_line.set_visible(True)
        else:
            final_line.set_visible(False)

        img.value = fig_png_bytes()
        status.value = format_info_block(i, k)

    slider.observe(lambda ch: render(ch["new"]), names="value")
    render(0)

    box = W.VBox([W.HBox([play, slider]), status, img])
    box.layout = W.Layout(width=f"{img_px}px")

    return box


### End-to-end grid demo

Try toggling:
- `map_type`: `'city'` vs `'maze'`
- `algo`: `'bfs'`, `'dfs'`, `'dijkstra'`, `'astar'`
- `weighted=True`: makes BFS and Dijkstra visibly different


In [ ]:
def run_grid_demo(
    map_type="city", algo="astar", weighted=False,
    h=61, w=61, seed=3, record_every=5,
    interval_ms=180, img_px=540, dpi=90,
    show=None,  # None => auto: show when weighted=True
):
    import numpy as np

    if map_type == "maze":
        grid = gen_maze(h, w, seed=seed)
    elif map_type == "city":
        grid = gen_manhattan_city(h, w, block=7, road=2, alley_prob=0.08, seed=seed)
    else:
        raise ValueError("map_type must be 'maze' or 'city'")

    start, goal = sample_start_goal(grid, seed=seed+1)

    cell_cost = None
    if weighted:
        cell_cost = make_costs(grid, seed=seed+2, p=0.18, lo=2, hi=8)
        cell_cost = np.where(grid == 1, np.nan, cell_cost)  # use nan for buildings for cleaner overlay

    steps, explored_order, came_from, found, stats = search_grid_with_steps(
        grid, start, goal, algo=algo, cell_cost=cell_cost,
        record_every=record_every, frontier_sample_n=1200
    )
    path = reconstruct_path(came_from, start, goal) if found else None
    pc = path_cost(path, cell_cost=cell_cost)

    # Auto show rule: show=True when weighted=True (unless explicitly overridden)
    if show is None:
        show = bool(weighted)

    info = {
        "map": map_type.upper(),
        "algo": algo.upper(),
        "weighted": weighted,
        "found": found,
        "expanded_total": stats.get("expanded"),
        "time_s": f"{stats.get('time_s', 0.0):.4f}",
        "path_cost": f"{pc:.1f}" if pc is not None else None,
        "path_len": (len(path) if path else None),
        "record_every": record_every,
        "frontier_sample_n": 1200,
        "start": start,
        "goal": goal,
    }

    if cell_cost is not None:
        info["traffic_min"] = float(np.nanmin(cell_cost))
        info["traffic_max"] = float(np.nanmax(cell_cost))

    box = interactive_grid_player(
        grid, start, goal, steps, explored_order,
        path=path,
        title="",
        info=info,
        cell_cost=cell_cost,        # traffic auto-on because cell_cost != None
        interval_ms=interval_ms,
        explored_tail=7000,
        dpi=dpi,
        img_px=img_px,
        explored_alpha=0.60,
        frontier_alpha=0.90,
        explored_color="tab:blue",
        frontier_color="gold",
        show=show,
    )
    return box


# Example runs:
run_grid_demo(map_type="city", algo="dijkstra", weighted=True, seed=233, interval_ms=220)


In [ ]:
def build_grid_demo_widget(map_type="city", algo="astar", weighted=False,
                           h=61, w=61, seed=3, record_every=8,
                           interval_ms=220, img_px=420, dpi=80):
    if map_type == "maze":
        grid = gen_maze(h, w, seed=seed)
    else:
        grid = gen_manhattan_city(h, w, block=7, road=2, alley_prob=0.08, seed=seed)

    start, goal = sample_start_goal(grid, seed=seed+1)

    cell_cost = None
    if weighted:
        cell_cost = make_costs(grid, seed=seed+2, p=0.18, lo=2, hi=8)
        cell_cost = np.where(grid == 1, 0.0, cell_cost)

    steps, explored_order, came_from, found, stats = search_grid_with_steps(
        grid, start, goal, algo=algo, cell_cost=cell_cost,
        record_every=record_every, frontier_sample_n=1200
    )
    path = reconstruct_path(came_from, start, goal) if found else None
    pc = path_cost(path, cell_cost=cell_cost)

    info = {
        "algo": algo.upper(),
        "found": found,
        "expanded": stats.get("expanded"),
        "time_s": f"{stats.get('time_s',0.0):.4f}",
        "path_cost": f"{pc:.1f}" if pc is not None else None,
        "path_len": len(path) if path else None,
    }

    # IMPORTANT: return the widget, don't rely on scrolling outputs
    return interactive_grid_player(
        grid, start, goal, steps, explored_order, path=path,
        title="", info=info,
        interval_ms=interval_ms, explored_tail=7000,
        dpi=dpi, img_px=img_px,
        explored_alpha=0.60, frontier_alpha=0.90,
        explored_color="tab:blue", frontier_color="gold"
    )


In [ ]:
def compare_algos_grid(
    algos=("bfs","dfs","dijkstra","astar"),
    map_type="city", weighted=False, seed=233,
    cols=2, img_px=420, interval_ms=120
):
    # One shared environment per comparison (same grid/start/goal)
    # So results are comparable.
    if map_type == "maze":
        grid = gen_maze(61, 61, seed=seed)
    else:
        grid = gen_manhattan_city(61, 61, block=7, road=2, alley_prob=0.08, seed=seed)

    start, goal = sample_start_goal(grid, seed=seed+1)

    cell_cost = None
    if weighted:
        cell_cost = make_costs(grid, seed=seed+2, p=0.18, lo=2, hi=8)
        cell_cost = np.where(grid == 1, 0.0, cell_cost)

    boxes = []
    for algo in algos:
        steps, explored_order, came_from, found, stats = search_grid_with_steps(
            grid, start, goal, algo=algo, cell_cost=cell_cost,
            record_every=40, frontier_sample_n=1200
        )
        path = reconstruct_path(came_from, start, goal) if found else None
        pc = path_cost(path, cell_cost=cell_cost)

        info = {
            "algo": algo.upper(),
            "found": found,
            "expanded": stats.get("expanded"),
            "time_s": f"{stats.get('time_s',0.0):.4f}",
            "path_cost": f"{pc:.1f}" if pc is not None else None,
            "path_len": len(path) if path else None,
            "start": start,
            "goal": goal,
        }

        box = interactive_grid_player(
            grid, start, goal, steps, explored_order, path=path,
            title="", info=info,
            interval_ms=interval_ms, explored_tail=7000,
            dpi=80, img_px=img_px, cell_cost=cell_cost
        )
        boxes.append(box)

    gridbox = W.GridBox(
        boxes,
        layout=W.Layout(
            grid_template_columns=" ".join(["1fr"] * cols),
            grid_gap="12px"
        )
    )
    display(gridbox)

# Run:
compare_algos_grid(map_type="city", weighted=True, seed=625, cols=2, img_px=380, interval_ms=240)


### Quick comparison table (BFS vs Dijkstra vs A*)

This prints a small table so students can see when BFS and Dijkstra match (unit cost) and when they diverge (weighted).


In [ ]:
def compare_algorithms(map_type="city", weighted=False, seed=7, h=61, w=61):
    if map_type == "maze":
        grid = gen_maze(h, w, seed=seed)
    else:
        grid = gen_manhattan_city(h, w, block=7, road=2, alley_prob=0.08, seed=seed)

    start, goal = sample_start_goal(grid, seed=seed+1)

    cell_cost = None
    if weighted:
        cell_cost = make_costs(grid, seed=seed+2, p=0.18, lo=2, hi=8)
        cell_cost = np.where(grid == 1, 0.0, cell_cost)

    rows = []
    for algo in ["bfs", "dfs", "dijkstra", "astar"]:
        steps, explored_order, came_from, found, stats = search_grid_with_steps(
            grid, start, goal, algo=algo, cell_cost=cell_cost,
            record_every=10**9  # no snapshots
        )
        path = reconstruct_path(came_from, start, goal) if found else None
        rows.append({
            "algo": algo,
            "found": found,
            "expanded": stats["expanded"],
            "time_s": round(stats["time_s"], 4),
            "path_len": (len(path) if path else None),
            "path_cost": (round(path_cost(path, cell_cost), 1) if path else None),
        })

    # Pretty display if pandas exists, else print rows
    try:
        import pandas as pd
        df = pd.DataFrame(rows).sort_values(["path_cost", "expanded"], na_position="last")
        display(df)
    except Exception:
        for r in rows:
            print(r)

# Try:
compare_algorithms(map_type="city", weighted=False, seed=7)
compare_algorithms(map_type="city", weighted=True, seed=7)

## Part B (optional) — Real Manhattan street network (OSMnx)

This section is heavier and depends on `osmnx`.

Key points:
- Use a **projected graph** so nearest-node lookup does **not** require scikit-learn BallTree.
- Restrict to a **small bbox subgraph** for interactive playback.

If you only care about teaching search concepts, the grid city above is usually better (cleaner control of heuristics and costs).


In [ ]:
# Optional: real Manhattan with OSMnx.
# If you don't have osmnx installed, either install it or skip this section.

try:
    import osmnx as ox
    import networkx as nx
    from shapely.geometry import Point
    from matplotlib.collections import LineCollection
    HAVE_OSMNX = True
except Exception as e:
    HAVE_OSMNX = False
    print("OSMnx not available:", repr(e))

In [ ]:
def build_osm_graph(place="Manhattan, New York City, New York, USA", network_type="walk"):
    if not HAVE_OSMNX:
        raise RuntimeError("OSMnx is not installed.")
    G = ox.graph_from_place(place, network_type=network_type, simplify=True)
    Gp = ox.project_graph(G)  # meters
    return Gp

def nearest_node_projected(Gp, latlon):
    pt, _ = ox.projection.project_geometry(Point(latlon[1], latlon[0]), to_crs=Gp.graph["crs"])
    return ox.distance.nearest_nodes(Gp, X=pt.x, Y=pt.y), (pt.x, pt.y)

def subgraph_bbox(Gp, xy_a, xy_b, pad_m=2500):
    xa, ya = xy_a
    xb, yb = xy_b
    xmin, xmax = (min(xa, xb) - pad_m, max(xa, xb) + pad_m)
    ymin, ymax = (min(ya, yb) - pad_m, max(ya, yb) + pad_m)

    nodes = ox.graph_to_gdfs(Gp, nodes=True, edges=False)
    keep = nodes[(nodes["x"] >= xmin) & (nodes["x"] <= xmax) & (nodes["y"] >= ymin) & (nodes["y"] <= ymax)].index
    H = Gp.subgraph(keep).copy()

    # keep largest weakly connected component to reduce no-path surprises
    comps = list(nx.weakly_connected_components(H))
    if comps:
        H = H.subgraph(max(comps, key=len)).copy()
    return H

In [ ]:
def euclid_h(G, u, v):
    ux, uy = G.nodes[u]["x"], G.nodes[u]["y"]
    vx, vy = G.nodes[v]["x"], G.nodes[v]["y"]
    return float(((ux - vx)**2 + (uy - vy)**2) ** 0.5)

def search_graph_with_steps(G, start, goal, algo="astar", weight="length",
                            record_every=80, frontier_sample_n=600, max_expand=300000):
    if algo not in ("dijkstra", "astar"):
        raise ValueError("algo must be 'dijkstra' or 'astar'")

    came_from = {}
    g = {start: 0.0}
    closed = set()
    open_best = {start: 0.0}
    heap = []
    tie = 0
    expanded = 0
    steps = []
    visited_order = []

    def f(node, g_cost):
        if algo == "dijkstra":
            return g_cost
        return g_cost + euclid_h(G, node, goal)

    heapq.heappush(heap, (f(start, 0.0), tie, start))

    while heap and expanded < max_expand:
        _, _, cur = heapq.heappop(heap)
        if cur in closed:
            continue
        if cur not in open_best:
            continue

        del open_best[cur]
        closed.add(cur)
        visited_order.append(cur)
        expanded += 1

        if expanded % record_every == 0:
            fs = list(open_best.keys())
            if len(fs) > frontier_sample_n:
                fs = fs[:frontier_sample_n]
            steps.append({"k": len(visited_order), "cur": cur, "frontier": fs})

        if cur == goal:
            steps.append({"k": len(visited_order), "cur": cur, "frontier": list(open_best.keys())[:frontier_sample_n]})
            return steps, visited_order, came_from, True, expanded

        for nb in G.neighbors(cur):
            if nb in closed:
                continue
            data = G.get_edge_data(cur, nb)
            if data is None:
                continue
            # MultiDiGraph: pick min edge weight
            best_w = None
            for _, attrs in data.items():
                w = attrs.get(weight, 1.0)
                if best_w is None or w < best_w:
                    best_w = w
            if best_w is None:
                best_w = 1.0

            new_g = g[cur] + float(best_w)
            if nb not in g or new_g < g[nb]:
                g[nb] = new_g
                came_from[nb] = cur
                tie += 1
                fs = f(nb, new_g)
                heapq.heappush(heap, (fs, tie, nb))
                open_best[nb] = fs

    steps.append({"k": len(visited_order), "cur": visited_order[-1] if visited_order else start,
                  "frontier": list(open_best.keys())[:frontier_sample_n]})
    return steps, visited_order, came_from, False, expanded

In [ ]:
def reconstruct_graph(came_from, start, goal):
    if start == goal:
        return [start]
    if goal not in came_from:
        return None
    cur = goal
    path = [cur]
    while cur != start:
        cur = came_from[cur]
        path.append(cur)
    path.reverse()
    return path

### Smooth OSMnx playback

We draw all edges once as a `LineCollection`, then only update node scatters.

If this is still slow:
- reduce `pad_m`
- increase `record_every`
- reduce `frontier_sample_n`


In [ ]:
def interactive_osm_player(G, start, goal, steps, visited_order, path=None,
                           title="", interval_ms=120, explored_tail=8000, dpi=90, figsize=(5.5, 5.5), img_px=540):
    # Flicker-free playback for OSMnx graphs using a widgets.Image (PNG updates).
    if W is None:
        raise RuntimeError("ipywidgets is not installed. Install with: pip install -q ipywidgets")

    import io
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

    fig = Figure(figsize=figsize, dpi=dpi)
    FigureCanvas(fig)
    ax = fig.add_subplot(1, 1, 1)
    ax.set_xticks([]); ax.set_yticks([])

    # Build edge segments once
    segs = []
    for u, v, data in G.edges(data=True):
        geom = data.get("geometry", None)
        if geom is not None:
            xs, ys = geom.xy
            segs.append(np.c_[xs, ys])
        else:
            x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
            x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]
            segs.append(np.array([[x1, y1], [x2, y2]]))

    lc = LineCollection(segs, linewidths=0.4, alpha=0.18)
    ax.add_collection(lc)
    ax.autoscale()

    explored_sc = ax.scatter([], [], s=10, alpha=0.25)
    frontier_sc = ax.scatter([], [], s=10, alpha=0.55)
    cur_sc      = ax.scatter([], [], s=50, alpha=1.0, edgecolors="k", linewidths=0.7)
    path_sc     = ax.scatter([], [], s=14, alpha=0.9) if path else None

    ax.scatter([G.nodes[start]["x"]], [G.nodes[start]["y"]], s=70, edgecolors="k", linewidths=0.7)
    ax.scatter([G.nodes[goal]["x"]],  [G.nodes[goal]["y"]],  s=70, edgecolors="k", linewidths=0.7)

    def nodes_xy(nodes):
        if not nodes:
            return np.empty((0, 2))
        xs = [G.nodes[n]["x"] for n in nodes]
        ys = [G.nodes[n]["y"] for n in nodes]
        return np.c_[xs, ys]

    def fig_png_bytes():
        buf = io.BytesIO()
        fig.canvas.draw()
        fig.canvas.print_png(buf)
        return buf.getvalue()

    slider = W.IntSlider(value=0, min=0, max=max(0, len(steps)-1), step=1, description="frame")
    play = W.Play(value=0, min=0, max=slider.max, step=1, interval=interval_ms)
    W.jslink((play, "value"), (slider, "value"))

    img = W.Image(value=b"", format="png")
    img.layout = W.Layout(width=f"{img_px}px", height=f"{img_px}px")
    status = W.HTML()

    def render(i):
        s = steps[i]
        k = s["k"]
        cur = s["cur"]
        frontier = s["frontier"]

        explored = visited_order[:k]
        if explored_tail and len(explored) > explored_tail:
            explored = explored[-explored_tail:]

        explored_sc.set_offsets(nodes_xy(explored))
        frontier_sc.set_offsets(nodes_xy(frontier))
        cur_sc.set_offsets(nodes_xy([cur]))
        if path_sc is not None:
            path_sc.set_offsets(nodes_xy(path))

        ax.set_title(f"{title}  frame={i}/{len(steps)-1}  expanded~{k}")
        img.value = fig_png_bytes()
        status.value = f"<b>{title}</b><br/>frame={i}/{len(steps)-1}, expanded~{k}"

    def on_change(change):
        render(change["new"])

    slider.observe(on_change, names="value")
    render(0)

    display(W.VBox([W.HBox([play, slider]), status, img]))
    return img


In [ ]:
def run_osm_demo(
    orig_latlon=(40.7484, -73.9857),
    dest_latlon=(40.7061, -74.0087),
    algo="astar",
    pad_m=2500,
    record_every=120,
    frontier_sample_n=600,
):
    if not HAVE_OSMNX:
        raise RuntimeError("Install osmnx to run this section.")

    place = "Manhattan, New York City, New York, USA"
    Gp = build_osm_graph(place, network_type="walk")

    orig, orig_xy = nearest_node_projected(Gp, orig_latlon)
    dest, dest_xy = nearest_node_projected(Gp, dest_latlon)

    H = subgraph_bbox(Gp, orig_xy, dest_xy, pad_m=pad_m)
    if orig not in H or dest not in H:
        H = Gp  # fallback

    steps, visited_order, came_from, found, expanded = search_graph_with_steps(
        H, orig, dest, algo=algo, weight="length",
        record_every=record_every, frontier_sample_n=frontier_sample_n
    )
    path = reconstruct_graph(came_from, orig, dest) if found else None

    title = f"OSMnx Manhattan | {algo.upper()} | found={found} | nodes={len(H)} | expanded={expanded}"
    interactive_osm_player(H, orig, dest, steps, visited_order, path=path, title=title)

# Example (optional):
run_osm_demo(algo="astar", pad_m=2200, record_every=160)